# E11: Final DEV Models and OOF Predictions (Models A, B, C)

**Spec sections**: 5.5 (mandatory: binning refitted in every fold), 7.3, 7.4, 7.5, 7.6.  
**Data**: DEV GOOD+BAD only; `params/chosen_params.json` (E10); R-folds repeat 1 and T-folds T1–T3 (E3), joined on app_id.  
**Outputs**: `models/A|B|C/*` + `models/sha256.json`, `oof_predictions.parquet`, `reports/04_training.xlsx`, `reports/04_ebm_shapes.png`, `reports/run_summaries/s04_train.json`, MLflow run.

1. Final models on all DEV: A = BinningProcess -> WoE -> unpenalised LR (sklearn and statsmodels, Scorecard points); B = EBM (interactions 0, monotonised where allowed); C = monotone LightGBM with the fixed final rounds.
2. OOF on R-fold repeat 1: every fold refitted from scratch (A includes binning; B/C include prepare_raw).
3. T-fold AUC per model (A refits binning per T-fold); EBM shape check on the T1–T3 training fits.

In [0]:
%pip install optbinning "numpy<2" "pyarrow==14.0.1" "statsmodels>=0.14.4" lightgbm interpret-core "scipy>=1.12,<1.14" --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, json, math, time, pickle, shutil, tempfile, warnings, importlib, hashlib
from datetime import datetime
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
importlib.reload(common)
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
import lightgbm as lgb
import interpret
from interpret.glassbox import ExplainableBoostingClassifier
import optbinning
from optbinning import BinningProcess, Scorecard
import statsmodels
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Missing values detected")   # EBM notice; NaN handled natively

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
SEED = cfg["random_seed"]
np.random.seed(SEED)
TUNE = cfg["tuning"]
N_THREADS = cfg["runtime"]["lgb_num_threads"]
PVALUE_MAX = cfg["funnel"]["L5"]["pvalue_max"]
SCALING = cfg["scaling"]

ARTIFACTS  = common.artifacts_path(cfg)
REPORTS    = os.path.join(ARTIFACTS, "reports")
RUN_SUMS   = os.path.join(REPORTS, "run_summaries")
FOLDS_DIR  = os.path.join(ARTIFACTS, "folds")
FEAT_LISTS = os.path.join(ARTIFACTS, "feature_lists")
BIN_DIR    = os.path.join(ARTIFACTS, "binning", "L1")
PARAMS_DIR = os.path.join(ARTIFACTS, "params")
MODELS_DIR = os.path.join(ARTIFACTS, "models")
OOF_PATH   = os.path.join(ARTIFACTS, "oof_predictions.parquet")
for d in (REPORTS, RUN_SUMS, *(os.path.join(MODELS_DIR, m) for m in "ABC")):
    os.makedirs(d, exist_ok=True)

cat, sch = cfg["databricks"]["catalog"], cfg["databricks"]["schema"]
MASTER_TBL = f"{cat}.{sch}.acard_master"
id_col, date_col = cfg["columns"]["id"], cfg["columns"]["date"]
target_col, status_col = cfg["columns"]["target"], cfg["columns"]["status"]

GIT_COMMIT = common.git_commit(PROJECT_ROOT)
CONFIG_HASH = common.config_hash(cfg)
VERSIONS = {"sklearn": sklearn.__version__, "lightgbm": lgb.__version__, "interpret": interpret.__version__,
            "optbinning": optbinning.__version__, "statsmodels": statsmodels.__version__,
            "numpy": np.__version__, "pandas": pd.__version__}

# WoE value given to missing / special rows in Model A. OptBinning default (0), the convention used in E9 (L5)
# and E10. Open question [E11-woe-missing] asks the model owner whether the empirical bin WoE should be used.
WOE_KW = {"metric_special": 0, "metric_missing": 0}

def rnd(x):
    """Round half up (avoid Python banker's rounding)."""
    return int(math.floor(x + 0.5))

_TMP = tempfile.mkdtemp(prefix="e11_")
_t0 = time.time()
print(f"Project root: {PROJECT_ROOT}")
print(f"Git commit  : {GIT_COMMIT[:12]} | Config hash: {CONFIG_HASH[:12]}")
print(f"Versions    : {VERSIONS}")
print(f"Scaling     : {SCALING} | p-value max {PVALUE_MAX} | lgb_num_threads={N_THREADS}")

In [0]:
# ── DEV only: Spark filter then common.load_window (OOT never read) ──
master_dev = spark.table(MASTER_TBL).filter("window = 'DEV'").toPandas()
assert set(master_dev["window"].unique()) == {"DEV"}, "non-DEV rows loaded"
oot_s, oot_e = cfg["windows"]["OOT"]
_d_all = pd.to_datetime(master_dev[date_col])
N_OOT_ROWS_READ = int(((_d_all >= oot_s) & (_d_all < oot_e)).sum())
assert N_OOT_ROWS_READ == 0, "OOT-dated rows present in the loaded data"

dev = common.load_window(master_dev, "DEV", cfg)
dev_gb = dev[dev[status_col].isin(cfg["labelled_statuses"])]
# Keep the DEV population frozen at E3 (folds/r_folds.parquet); later appends to the master table
# (FWD/STRESS rebuild) added DEV records that the funnel, tuning and folds were never built on
frozen_ids = set(pd.read_parquet(os.path.join(FOLDS_DIR, "r_folds.parquet"), columns=["app_id"])["app_id"].astype(str))
n_new = int((~dev_gb[id_col].astype(str).isin(frozen_ids)).sum())
print(f"DEV GOOD+BAD records not in E3 folds (excluded): {n_new}")
dev_gb = dev_gb[dev_gb[id_col].astype(str).isin(frozen_ids)].reset_index(drop=True)
common.assert_labelled(dev_gb, cfg)
y = (dev_gb[status_col] == "BAD").astype(int)
exp = cfg["expected_counts"]["DEV"]
assert len(dev_gb) == exp["records"] and int(y.sum()) == exp["bads"], "DEV counts differ from expected_counts"
print(f"DEV GOOD+BAD: {len(dev_gb):,} rows, {int(y.sum())} Bads (expected {exp['records']:,}/{exp['bads']})")

dev_gb["app_id"] = dev_gb[id_col].astype(str)
assert dev_gb["app_id"].is_unique, "app_id not unique in DEV GOOD+BAD — fold join would be ambiguous"
pos = pd.Series(np.arange(len(dev_gb)), index=dev_gb["app_id"])

def build_folds(fdf, keys):
    out = []
    for key, g in fdf.groupby(keys, sort=True):
        if isinstance(key, tuple) and len(key) == 1:
            key = key[0]
        tr = pos.reindex(g.loc[g["part"] == "train", "app_id"].astype(str))
        va = pos.reindex(g.loc[g["part"] == "valid", "app_id"].astype(str))
        assert tr.notna().all() and va.notna().all(), f"fold {key}: app_id not in DEV GOOD+BAD"
        tr, va = tr.to_numpy(int), va.to_numpy(int)
        assert len(np.intersect1d(tr, va)) == 0, f"fold {key}: train/valid overlap"
        out.append((key, np.sort(tr), np.sort(va)))
    return out

rf = pd.read_parquet(os.path.join(FOLDS_DIR, "r_folds.parquet"))
tf = pd.read_parquet(os.path.join(FOLDS_DIR, "t_folds.parquet"))
assert set(rf["app_id"].astype(str)) == set(dev_gb["app_id"]), "R-fold ids != DEV GOOD+BAD ids"
assert set(tf["app_id"].astype(str)) <= set(dev_gb["app_id"]), "T-fold ids outside DEV GOOD+BAD"
R = build_folds(rf, ["repeat", "fold"])
T = build_folds(tf, ["fold"])
R1 = [(k, tr, va) for k, tr, va in R if k[0] == 0]          # spec 7.6: first repeat only
assert len(R1) == cfg["cv"]["r_folds"]["n_splits"]
r1_valid = np.concatenate([va for _, _, va in R1])
assert len(r1_valid) == len(dev_gb) and len(np.unique(r1_valid)) == len(dev_gb), "R1 valid parts do not partition DEV"
assert [k for k, _, _ in T] == sorted(cfg["cv"]["t_folds"].keys())
print(f"R-fold repeat 1: {len(R1)} folds, valid sizes {[len(va) for _, _, va in R1]}")
for k, tr, va in T:
    print(f"  {k}: train {len(tr):,} ({int(y.iloc[tr].sum())} Bads)  valid {len(va):,} ({int(y.iloc[va].sum())} Bads)")

with open(os.path.join(FEAT_LISTS, "model_a.json")) as f:
    A_FEATS = [r["feature"] for r in json.load(f)["features"]]
with open(os.path.join(FEAT_LISTS, "l4_pool.json")) as f:
    POOL = [r["feature"] for r in json.load(f)]
fdict = pd.read_csv(os.path.join(PROJECT_ROOT, "config", "feature_dict.csv"))
fdx = fdict.set_index("feature")

with open(os.path.join(PARAMS_DIR, "chosen_params.json")) as fh:
    CP = json.load(fh)
assert A_FEATS == CP["A"]["features"], "model_a.json differs from the E10 Model A feature list"
assert POOL == CP["B"]["input_features"] == CP["C"]["input_features"], "l4_pool.json differs from the E10 B/C inputs"
if CP["config_sha256"] != CONFIG_HASH:
    print(f"WARNING: config hash changed since E10 ({CP['config_sha256'][:12]} -> {CONFIG_HASH[:12]})")
VER_DIFF = {k: (CP["versions"].get(k), v) for k, v in VERSIONS.items() if k in CP["versions"] and CP["versions"][k] != v}
print(f"Model A features: {len(A_FEATS)} | L4 pool (B, C): {len(POOL)} | E10 trials: B {CP['B']['trial']}, C {CP['C']['trial']}")
print(f"Library versions differing from E10: {VER_DIFF or 'none'}")

In [0]:
# ── L1 binning objects: trend for B monotonisation / C constraints, and L1 bins for the B shape check ──
binning_objects = {}
for f in POOL:
    with open(os.path.join(BIN_DIR, f"{f}.pkl"), "rb") as fh:
        binning_objects[f] = pickle.load(fh)
ALL = np.arange(len(dev_gb))

def x_raw(idx):
    """Raw matrix for B and C (special codes -> NaN + indicator), built from the given rows only."""
    return common.prepare_raw(dev_gb.iloc[idx], POOL, fdict)

X_RAW_ALL = x_raw(ALL)
COLS = list(X_RAW_ALL.columns)
assert COLS == CP["B"]["columns"] == CP["C"]["columns"], "prepare_raw columns differ from E10"
CAT_COLS = CP["C"]["categorical_feature"]
MONO = CP["C"]["monotone_constraints"]
assert MONO == common.monotone_vector(COLS, POOL, binning_objects, fdict), "monotone vector differs from E10"

# ── Model A: BinningProcess with the L1 parameters and special codes saved by E10 ──
L1_BIN = CP["A"]["binning_params"]
A_CAT = CP["A"]["categorical_features"]
A_SPECIAL = CP["A"]["special_codes"]
A_FIT_PARAMS = {f: {**{k: v for k, v in L1_BIN.items() if k != "cat_cutoff" or f in A_CAT},
                    "special_codes": (A_SPECIAL[f] or None)} for f in A_FEATS}

def x_a(idx):
    X = dev_gb.iloc[idx][A_FEATS].copy()
    for f in A_FEATS:
        if f not in A_CAT:
            X[f] = pd.to_numeric(X[f], errors="coerce")
    return X

def make_bp():
    return BinningProcess(variable_names=A_FEATS, categorical_variables=A_CAT or None,
                          binning_fit_params=A_FIT_PARAMS)

# ── Model B: monotonise where expected_direction is + / - and agrees with the L1 trend (spec 7.4) ──
B_PARAMS = dict(CP["B"]["params"])
assert B_PARAMS["interactions"] == 0, "Model B must have interactions = 0"
DIR_TO_TREND = {"+": "ascending", "-": "descending"}
mono_rows = []
for c in COLS:
    if c not in POOL:
        mono_rows.append({"column": c, "expected_direction": None, "l1_trend": "indicator", "applied": False,
                          "increasing": None, "reason": "special-value indicator"})
        continue
    ed = str(fdx.loc[c, "expected_direction"]).strip()
    trd = common.binning_trend(binning_objects[c])
    ok = ed in DIR_TO_TREND and DIR_TO_TREND[ed] == trd and str(fdx.loc[c, "dtype"]).lower() != "categorical"
    reason = ("applied" if ok else "expected_direction is not + or -" if ed not in DIR_TO_TREND
              else f"L1 trend {trd} disagrees with expected_direction {ed}")
    mono_rows.append({"column": c, "expected_direction": ed, "l1_trend": trd, "applied": ok,
                      "increasing": (ed == "+") if ok else None, "reason": reason})
B_MONO_DF = pd.DataFrame(mono_rows)
# increasing=True: term score (log-odds of Bad) rises with the feature value, i.e. '+' / ascending Bad Rate
B_MONO = {r.column: bool(r.increasing) for r in B_MONO_DF.itertuples() if r.applied}

# ── Model C: chosen params + monotone vector; rounds scale with training size (no early stopping) ──
C_PARAMS = {**CP["C"]["params"], "monotone_constraints": MONO}
assert C_PARAMS["num_threads"] == N_THREADS and C_PARAMS["deterministic"] and C_PARAMS["seed"] == SEED
C_FINAL_ROUNDS = int(CP["C"]["final_rounds"])
C_BEST_ITER = int(CP["C"]["stage3_best_iter"])
R1_TRAIN_SIZE = float(np.mean([len(tr) for _, tr, _ in R1]))

def c_rounds(n_train):
    """Fold rounds = stage-3 best_iter (found at R1 training size) scaled to the training size."""
    return max(1, rnd(C_BEST_ITER * n_train / R1_TRAIN_SIZE))

# ── Fit log: the app_ids that entered every fit (checked against each fold's validation ids) ──
FIT_LOG = []
def _log(model, fold, idx):
    FIT_LOG.append({"model": model, "fold": fold, "n": len(idx), "ids": set(dev_gb["app_id"].iloc[idx])})

def fit_a(idx, fold):
    _log("A", fold, idx)
    pipe = Pipeline([("binning", make_bp()), ("lr", common.lr_unpenalised(max_iter=2000))])
    return pipe.fit(x_a(idx), y.iloc[idx])

def pred_a(m, idx):
    return m.predict_proba(x_a(idx))[:, 1]

def a_binning_counts(bp):
    """Number of records each fitted binning saw (Totals row of the binning table)."""
    return {f: int(bp.get_binned_variable(f).binning_table.build().loc["Totals", "Count"]) for f in A_FEATS}

def fit_b(idx, fold):
    _log("B", fold, idx)
    m = ExplainableBoostingClassifier(**B_PARAMS).fit(x_raw(idx), y.iloc[idx])
    for f, inc in B_MONO.items():
        m.monotonize(f, increasing=inc)
    return m

def pred_b(m, idx):
    return m.predict_proba(x_raw(idx))[:, 1]

def fit_c(idx, fold, rounds):
    _log("C", fold, idx)
    ds = lgb.Dataset(x_raw(idx), label=y.iloc[idx], categorical_feature=CAT_COLS, free_raw_data=False,
                     params={"feature_pre_filter": False, "verbose": -1})
    return lgb.train(C_PARAMS, ds, num_boost_round=rounds)

def pred_c(m, idx):
    return m.predict(x_raw(idx))

def perf(yy, p):
    auc = roc_auc_score(yy, p)
    return {"auc": auc, "gini": 2 * auc - 1, "ks": common.ks_stat(np.asarray(yy), np.asarray(p))}

print(f"Raw matrix {X_RAW_ALL.shape} | B monotonised: {len(B_MONO)} of {len(POOL)} "
      f"(expected_direction values: {B_MONO_DF.expected_direction.value_counts(dropna=False).to_dict()})")
print(f"C final rounds {C_FINAL_ROUNDS}; fold rounds at R1 train size {R1_TRAIN_SIZE:.0f} = {c_rounds(R1_TRAIN_SIZE)}")
display(B_MONO_DF)

In [0]:
# dev_gb.aecb_file_age_months.plot(kind='kde')
binning_objects["aecb_file_age_months"]

In [0]:
# ── Model A: Scorecard(BinningProcess + unpenalised LR) on all DEV; statsmodels Logit on the same WoE ──
X_A_ALL = x_a(ALL)
assert SCALING["odds_type"] == "good_to_bad", "pdo_odds scaling assumes Good:Bad odds (spec 9.3)"
SC_PARAMS = {"pdo": SCALING["pdo"], "odds": SCALING["base_odds"], "scorecard_points": SCALING["base_score"]}
_log("A", "final", ALL)
A_SC = Scorecard(binning_process=make_bp(), estimator=common.lr_unpenalised(max_iter=2000),
                 scaling_method="pdo_odds", scaling_method_params=SC_PARAMS,
                 reverse_scorecard=False, rounding=False)
A_SC.fit(X_A_ALL, y, **WOE_KW)
A_BP, A_LR = A_SC.binning_process_, A_SC.estimator_
assert list(A_BP.get_support(names=True)) == A_FEATS, "binning process dropped or reordered features"
X_WOE = pd.DataFrame(np.asarray(A_BP.transform(X_A_ALL, metric="woe", **WOE_KW), dtype=float), columns=A_FEATS)

A_LOGIT = sm.Logit(y.to_numpy(), sm.add_constant(X_WOE, has_constant="add")).fit(disp=0, maxiter=200)
A_LOGIT_CONVERGED = bool(A_LOGIT.mle_retvals["converged"])
coef_sk = pd.Series(np.r_[A_LR.intercept_[0], A_LR.coef_[0]], index=["const"] + A_FEATS)
A_COEF_DIFF = float((coef_sk - A_LOGIT.params).abs().max())
Xc = sm.add_constant(X_WOE, has_constant="add").to_numpy()
A_VIF = {f: float(variance_inflation_factor(Xc, i + 1)) for i, f in enumerate(A_FEATS)}
A_COEF = pd.DataFrame({
    "term": ["const"] + A_FEATS,
    "coef_sklearn": coef_sk.values,
    "coef_statsmodels": A_LOGIT.params.values,
    "abs_diff": (coef_sk - A_LOGIT.params).abs().values,
    "std_err": A_LOGIT.bse.values,
    "z": A_LOGIT.tvalues.values,
    "p_value": A_LOGIT.pvalues.values,
    "vif": [np.nan] + [A_VIF[f] for f in A_FEATS],
})
feat_coefs = A_LOGIT.params[A_FEATS]
A_SIGNS_EQUAL = bool((feat_coefs < 0).all() or (feat_coefs > 0).all())
A_PMAX = float(A_LOGIT.pvalues[A_FEATS].max())

# ── Points table; check score = Offset + Factor x ln(Good:Bad odds) and higher score = lower risk ──
A_POINTS = A_SC.table(style="detailed")
p_a_full = A_LR.predict_proba(X_WOE)[:, 1]
FACTOR = SCALING["pdo"] / np.log(2)
OFFSET = SCALING["base_score"] - FACTOR * np.log(SCALING["base_odds"])
score_formula = OFFSET + FACTOR * np.log((1 - p_a_full) / p_a_full)
A_SCORE = np.asarray(A_SC.score(X_A_ALL), dtype=float)
A_SCORE_DIFF = float(np.max(np.abs(A_SCORE - score_formula)))
A_SC_PROBA_DIFF = float(np.max(np.abs(A_SC.predict_proba(X_A_ALL)[:, 1] - p_a_full)))
A_SCORE_RISK_CORR = float(pd.Series(A_SCORE).corr(pd.Series(p_a_full), method="spearman"))

# ── Model A bins vs the stored L1 bins (same parameters, same DEV data) ──
bin_rows = []
for f in A_FEATS:
    ob_a, ob_1 = A_BP.get_binned_variable(f), binning_objects[f]
    s_a, s_1 = np.asarray(ob_a.splits, dtype=float), np.asarray(ob_1.splits, dtype=float)
    bin_rows.append({"feature": f, "n_splits_A": len(s_a), "n_splits_L1": len(s_1),
                     "same_splits": len(s_a) == len(s_1) and bool(np.allclose(s_a, s_1)),
                     "trend_A": common.binning_trend(ob_a), "trend_L1": common.binning_trend(ob_1)})
A_BINS_VS_L1 = pd.DataFrame(bin_rows)

print(f"Model A: sklearn vs statsmodels max |coef diff| = {A_COEF_DIFF:.2e} | Logit converged: {A_LOGIT_CONVERGED}")
print(f"  signs equal: {A_SIGNS_EQUAL} ({int((feat_coefs < 0).sum())} negative of {len(A_FEATS)}) | "
      f"max p = {A_PMAX:.4g} (limit {PVALUE_MAX}) | max VIF = {max(A_VIF.values()):.2f}")
print(f"  score check: max |Scorecard.score - formula| = {A_SCORE_DIFF:.2e}; Spearman(score, PD) = {A_SCORE_RISK_CORR:.4f}; "
      f"score range {A_SCORE.min():.1f}-{A_SCORE.max():.1f}")
print(f"  bins identical to L1: {int(A_BINS_VS_L1.same_splits.sum())}/{len(A_FEATS)}; "
      f"trend agreement: {int((A_BINS_VS_L1.trend_A == A_BINS_VS_L1.trend_L1).sum())}/{len(A_FEATS)}")
display(A_COEF)

In [0]:
# ── Model B: EBM with the chosen params on all DEV ──
t = time.time()
B_FINAL = fit_b(ALL, "final")
assert all(len(tf_) == 1 for tf_ in B_FINAL.term_features_), "EBM contains interaction terms"
B_IMPORTANCE = (pd.DataFrame({"term": B_FINAL.term_names_, "importance_avg_weight": B_FINAL.term_importances()})
                .sort_values("importance_avg_weight", ascending=False).reset_index(drop=True))
print(f"Model B fitted in {time.time() - t:.0f}s: {len(B_FINAL.term_names_)} main-effect terms, "
      f"{len(B_MONO)} monotonised")

# ── Model C: LightGBM with the fixed final rounds, no early stopping ──
t = time.time()
C_FINAL = fit_c(ALL, "final", C_FINAL_ROUNDS)
assert C_FINAL.num_trees() == C_FINAL_ROUNDS, "LightGBM tree count differs from the final rounds"
C_IMPORTANCE = (pd.DataFrame({"feature": C_FINAL.feature_name(),
                              "gain": C_FINAL.feature_importance("gain"),
                              "split": C_FINAL.feature_importance("split"),
                              "monotone": MONO})
                .sort_values("gain", ascending=False).reset_index(drop=True))
print(f"Model C fitted in {time.time() - t:.0f}s: {C_FINAL.num_trees()} trees")

IN_SAMPLE = pd.DataFrame([{"model": "A", **perf(y, p_a_full)},
                          {"model": "B", **perf(y, pred_b(B_FINAL, ALL))},
                          {"model": "C", **perf(y, pred_c(C_FINAL, ALL))}])
print("In-sample DEV metrics (for reference only; OOF below is the honest DEV estimate):")
display(IN_SAMPLE)

In [0]:
# ── Spec 7.6: every fold fitted from scratch (A: binning + LR; B/C: prepare_raw + model) ──
oof_parts, oof_fold_rows, A_BIN_COUNT_CHECK = [], [], []
FOLD_VALID = {}
for (rep, fold), tr, va in R1:
    t = time.time()
    fname = f"R1_f{int(fold)}"
    ids_va = dev_gb["app_id"].iloc[va].to_numpy()
    FOLD_VALID[fname] = set(ids_va)
    ma = fit_a(tr, fname)
    A_BIN_COUNT_CHECK.append({"fold": fname, "n_train": len(tr),
                              "ok": all(c == len(tr) for c in a_binning_counts(ma.named_steps["binning"]).values())})
    mb = fit_b(tr, fname)
    rc = c_rounds(len(tr))
    mc = fit_c(tr, fname, rc)
    for model, p in (("A", pred_a(ma, va)), ("B", pred_b(mb, va)), ("C", pred_c(mc, va))):
        oof_parts.append(pd.DataFrame({"app_id": ids_va, "model": model, "fold": int(fold), "p_oof": p}))
        oof_fold_rows.append({"model": model, "fold": int(fold), "n_train": len(tr), "n_valid": len(va),
                              "rounds": rc if model == "C" else None, **perf(y.iloc[va], p)})
    print(f"  {fname}: train {len(tr):,} valid {len(va):,} | C rounds {rc} | {time.time() - t:.0f}s")

OOF = pd.concat(oof_parts, ignore_index=True)[["app_id", "model", "fold", "p_oof"]]
OOF["fold"] = OOF["fold"].astype(int)
y_by_id = pd.Series(y.to_numpy(), index=dev_gb["app_id"])
OOF_METRICS = pd.DataFrame([{"model": m, "n": len(g), "bads": int(y_by_id.loc[g.app_id].sum()),
                             **perf(y_by_id.loc[g.app_id].to_numpy(), g.p_oof.to_numpy())}
                            for m, g in OOF.groupby("model")])
OOF_FOLD = pd.DataFrame(oof_fold_rows)
_tmp = os.path.join(_TMP, "oof_predictions.parquet")
OOF.to_parquet(_tmp, index=False)
shutil.copyfile(_tmp, OOF_PATH)
print(f"Saved {OOF_PATH}: {len(OOF):,} rows")
display(OOF_METRICS)

In [0]:
# ── T1–T3 expanding-window folds; A refits binning in every T-fold (input to rule R3) ──
t_rows, B_TFITS = [], {}
for name, tr, va in T:
    t = time.time()
    FOLD_VALID[name] = set(dev_gb["app_id"].iloc[va])
    ma = fit_a(tr, name)
    A_BIN_COUNT_CHECK.append({"fold": name, "n_train": len(tr),
                              "ok": all(c == len(tr) for c in a_binning_counts(ma.named_steps["binning"]).values())})
    mb = fit_b(tr, name)
    B_TFITS[name] = mb
    rc = c_rounds(len(tr))
    mc = fit_c(tr, name, rc)
    for model, p in (("A", pred_a(ma, va)), ("B", pred_b(mb, va)), ("C", pred_c(mc, va))):
        t_rows.append({"model": model, "tfold": name, "n_train": len(tr), "n_valid": len(va),
                       "rounds": rc if model == "C" else None, **perf(y.iloc[va], p)})
    print(f"  {name}: train {len(tr):,} valid {len(va):,} | C rounds {rc} | {time.time() - t:.0f}s")

TFOLD = pd.DataFrame(t_rows)
TFOLD_SUMMARY = (TFOLD.groupby("model")["auc"]
                 .agg(t_auc_mean="mean", t_auc_std=lambda s: float(np.std(s, ddof=0)), n_folds="count")
                 .reset_index())
TFOLD_SUMMARY["e10_t_auc_mean"] = TFOLD_SUMMARY["model"].map({"B": CP["B"]["t_auc_mean"], "C": CP["C"]["t_auc_mean"]})
TFOLD_SUMMARY["std_ratio_vs_A"] = TFOLD_SUMMARY["t_auc_std"] / float(TFOLD_SUMMARY.set_index("model").loc["A", "t_auc_std"])
display(TFOLD.pivot(index="model", columns="tfold", values="auc"))
display(TFOLD_SUMMARY)

In [0]:
# ── Spec 7.4 shape check. Each fit's term score is averaged within the L1 regular bins (the same bins L2 uses
#    for head-tail checks); head = first bin, tail = last bin. FAIL if sign(tail - head) differs across T1–T3. ──
def l1_regular_bins(f):
    """L1 regular-bin index per DEV row; -1 for missing / special rows."""
    ob = binning_objects[f]
    x = pd.to_numeric(dev_gb[f], errors="coerce")
    idx = np.asarray(ob.transform(x, metric="indices", metric_special="empirical", metric_missing="empirical"))
    n_reg = len(ob.splits) + 1
    return np.where(idx < n_reg, idx, -1)

BIN_IDX = {c: (l1_regular_bins(c) if c in POOL else X_RAW_ALL[c].to_numpy().astype(int)) for c in COLS}
FITS = {**B_TFITS, "final": B_FINAL}
shape_rows = []
for fit_name, m in FITS.items():
    contrib = np.asarray(m.eval_terms(X_RAW_ALL))
    names = list(m.term_names_)
    for c in COLS:
        b = BIN_IDX[c]
        mask = b >= 0
        s = pd.DataFrame({"bin": b[mask], "score": contrib[mask, names.index(c)]}).groupby("bin")["score"].agg(["mean", "size"])
        for bi, r in s.iterrows():
            shape_rows.append({"fit": fit_name, "column": c, "bin": int(bi), "n": int(r["size"]), "mean_score": float(r["mean"])})
B_SHAPE_BINS = pd.DataFrame(shape_rows)

check_rows = []
for c in COLS:
    d = B_SHAPE_BINS[(B_SHAPE_BINS.column == c) & B_SHAPE_BINS.fit.isin(list(B_TFITS))]
    diffs = {}
    for fn, g in d.groupby("fit"):
        g = g.sort_values("bin")
        diffs[fn] = float(g.mean_score.iloc[-1] - g.mean_score.iloc[0]) if len(g) >= 2 else np.nan
    vals = [v for v in diffs.values() if not np.isnan(v)]
    check_rows.append({"column": c, "n_bins": int(d.bin.nunique()),
                       **{f"tail_minus_head_{k}": v for k, v in diffs.items()},
                       "head_tail_reversal": bool(vals and max(vals) > 0 and min(vals) < 0)})
B_SHAPE_CHECK = pd.DataFrame(check_rows)
EBM_SHAPE_CHECK = "FAIL" if B_SHAPE_CHECK.head_tail_reversal.any() else "PASS"
REVERSED = B_SHAPE_CHECK.loc[B_SHAPE_CHECK.head_tail_reversal, "column"].tolist()

def ebm_shape_table(m):
    """Final EBM term scores per EBM bin (missing / regular bins / unseen)."""
    rows = []
    for ti, feats in enumerate(m.term_features_):
        fi, name = feats[0], m.term_names_[ti]
        scores, w = np.asarray(m.term_scores_[ti], dtype=float), np.asarray(m.bin_weights_[ti], dtype=float)
        b = m.bins_[fi][0]
        if isinstance(b, dict):
            labels = ["missing"] + [str(k) for k, _ in sorted(b.items(), key=lambda kv: kv[1])] + ["unseen"]
            lows, highs = [None] * len(labels), [None] * len(labels)
        else:
            cuts = [float(v) for v in np.asarray(b, dtype=float)]
            labels = ["missing"] + [f"bin_{j}" for j in range(len(cuts) + 1)] + ["unseen"]
            lows = [None, None] + cuts + [None]
            highs = [None] + cuts + [None, None]
        for j, lab in enumerate(labels):
            rows.append({"term": name, "bin": lab, "lower": lows[j], "upper": highs[j],
                         "score": scores[j] if j < len(scores) else np.nan, "weight": w[j] if j < len(w) else np.nan})
    return pd.DataFrame(rows)
B_SHAPES_FINAL = ebm_shape_table(B_FINAL)

# ── Plot: one panel per column, T1–T3 training fits and the final fit ──
ncol = 6
nrow = math.ceil(len(COLS) / ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3 * nrow), squeeze=False)
for ax, c in zip(axes.flat, COLS):
    for fn, g in B_SHAPE_BINS[B_SHAPE_BINS.column == c].groupby("fit"):
        g = g.sort_values("bin")
        ax.plot(g["bin"], g["mean_score"], marker="o", ms=3, label=fn,
                lw=2 if fn == "final" else 1, ls="--" if fn == "final" else "-")
    rev = c in REVERSED
    ax.set_title(c[:38] + (" [REVERSAL]" if rev else ""), fontsize=8, color="red" if rev else "black")
    ax.axhline(0, color="grey", lw=0.5)
    ax.tick_params(labelsize=7)
for ax in list(axes.flat)[len(COLS):]:
    ax.axis("off")
axes.flat[0].legend(fontsize=7)
fig.suptitle("Model B (EBM): mean term score (log-odds of Bad) per L1 bin — T1–T3 training fits and final DEV fit", y=1.0)
fig.tight_layout()
SHAPE_PNG = os.path.join(REPORTS, "04_ebm_shapes.png")
_tmp = os.path.join(_TMP, "04_ebm_shapes.png")
fig.savefig(_tmp, dpi=110, bbox_inches="tight")
shutil.copyfile(_tmp, SHAPE_PNG)
plt.show()
print(f"ebm_shape_check = {EBM_SHAPE_CHECK} | head-tail reversals across T1–T3: {REVERSED or 'none'}")
display(B_SHAPE_CHECK)

In [0]:
# ── Sensitivity only (the models above keep WOE_KW): R1 OOF AUC of Model A with the empirical WoE of the
#    Missing / Special bins instead of 0. Binning refitted in every fold, as above. ──
EMP = {"metric_special": "empirical", "metric_missing": "empirical"}
sens_parts = []
for (rep, fold), tr, va in R1:
    fname = f"R1_f{int(fold)}"
    _log("A_sens", fname, tr)
    bp = make_bp().fit(x_a(tr), y.iloc[tr])
    lr = common.lr_unpenalised(max_iter=2000).fit(bp.transform(x_a(tr), metric="woe", **EMP), y.iloc[tr])
    p = lr.predict_proba(bp.transform(x_a(va), metric="woe", **EMP))[:, 1]
    sens_parts.append(pd.DataFrame({"app_id": dev_gb["app_id"].iloc[va].to_numpy(), "p": p}))
sens = pd.concat(sens_parts, ignore_index=True)
AUC_A_OOF = float(OOF_METRICS.set_index("model").loc["A", "auc"])
AUC_A_EMP = float(roc_auc_score(y_by_id.loc[sens.app_id].to_numpy(), sens.p.to_numpy()))

miss_rows = []
for f in A_FEATS:
    bt = A_BP.get_binned_variable(f).binning_table.build()
    mr = bt[bt["Bin"].astype(str) == "Missing"].iloc[0]
    miss_rows.append({"feature": f, "missing_count": int(mr["Count"]), "missing_bads": int(mr["Event"]),
                      "missing_bad_rate": float(mr["Event rate"]) if mr["Count"] else np.nan,
                      "missing_bin_woe": float(mr["WoE"]), "coef": float(A_LOGIT.params[f]),
                      "woe_used_in_model": WOE_KW["metric_missing"]})
A_MISSING = pd.DataFrame(miss_rows)
A_SENS = pd.DataFrame([{"variant": f"metric_missing={WOE_KW['metric_missing']} (models)", "r1_oof_auc": AUC_A_OOF},
                       {"variant": "metric_missing=empirical (sensitivity)", "r1_oof_auc": AUC_A_EMP}])
n_feat_missing = int((A_MISSING.missing_count > 0).sum())
print(f"Model A R1 OOF AUC: WoE 0 for missing {AUC_A_OOF:.4f} | empirical missing WoE {AUC_A_EMP:.4f} "
      f"| {n_feat_missing}/{len(A_FEATS)} features have missing rows in DEV")

# Record the open question once (idempotent). UC volumes do not support append mode: rewrite the file.
oq_path = os.path.join(REPORTS, "open_questions.md")
marker = "[E11-woe-missing]"
existing = open(oq_path).read() if os.path.exists(oq_path) else ""
if marker not in existing:
    with open(oq_path, "w") as fh:
        fh.write(existing + f"\n- {marker} E11 ({datetime.now():%Y-%m-%d}): E9 (L5), E10 and E11 transform Model A with "
                 f"OptBinning's default metric_missing=0 / metric_special=0, so rows with a missing value get WoE 0 "
                 f"(neutral points) instead of the Missing bin's own WoE. Spec 2.4 gives missing its own bin and 13.2 "
                 f"counts missing/special bins in max points. {n_feat_missing} of {len(A_FEATS)} Model A features have "
                 f"missing rows. R1 OOF AUC: {AUC_A_OOF:.4f} (WoE 0) vs {AUC_A_EMP:.4f} (empirical). Model owner to "
                 f"confirm which convention to use; switching means re-running E9-E11. Models B/C are not affected.\n")
    print("Open question written to open_questions.md")
display(A_MISSING)

In [0]:
# ── Save model files under models/ (write to local temp, then copy to the UC volume) and hash every file ──
SAVED = {}
def _save(rel, writer):
    tmp = os.path.join(_TMP, rel.replace("/", "__"))
    writer(tmp)
    dst = os.path.join(MODELS_DIR, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copyfile(tmp, dst)
    SAVED[rel] = dst

def _pickler(obj):
    def w(p):
        with open(p, "wb") as fh:
            pickle.dump(obj, fh)
    return w

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

_save("A/binning_process.pkl", _pickler(A_BP))
_save("A/lr_sklearn.pkl", _pickler(A_LR))
_save("A/logit_statsmodels.pkl", _pickler(A_LOGIT))
_save("A/scorecard.pkl", _pickler(A_SC))
_save("A/points_table.csv", lambda p: A_POINTS.to_csv(p, index=False))
_save("A/coefficients.csv", lambda p: A_COEF.to_csv(p, index=False))
_save("B/ebm.pkl", _pickler(B_FINAL))
_save("C/lgbm.txt", lambda p: C_FINAL.save_model(p))
_save("C/lgbm.pkl", _pickler(C_FINAL))
MODEL_SPEC = {
    "module": "E11_train", "run_time": datetime.now().isoformat(), "git_commit": GIT_COMMIT, "config_sha256": CONFIG_HASH,
    "versions": VERSIONS,
    "A": {"features": A_FEATS, "binning_fit_params": A_FIT_PARAMS, "woe_transform": WOE_KW,
          "scaling": SC_PARAMS, "factor": FACTOR, "offset": OFFSET},
    "B": {"input_features": POOL, "columns": COLS, "params": B_PARAMS, "monotonized": B_MONO, "e10_trial": CP["B"]["trial"]},
    "C": {"input_features": POOL, "columns": COLS, "params": {k: v for k, v in C_PARAMS.items() if k != "monotone_constraints"},
          "monotone_constraints": MONO, "num_boost_round": C_FINAL_ROUNDS, "e10_trial": CP["C"]["trial"]},
    "raw_preparation": "common.prepare_raw (special codes -> NaN + <feature>_is_special_<code>)",
}
_save("model_spec.json", lambda p: json.dump(MODEL_SPEC, open(p, "w"), indent=2,
                                             default=lambda o: o.item() if hasattr(o, "item") else str(o)))

FILE_SHA = pd.DataFrame([{"file": rel, "path": p, "bytes": os.path.getsize(p), "sha256": sha256_file(p)}
                         for rel, p in SAVED.items()])
SHA_PATH = os.path.join(MODELS_DIR, "sha256.json")
with open(SHA_PATH, "w") as fh:
    json.dump({r.file: r.sha256 for r in FILE_SHA.itertuples()}, fh, indent=2)
print(f"Saved {len(SAVED)} model files under {MODELS_DIR}; hashes in {SHA_PATH}")
display(FILE_SHA[["file", "bytes", "sha256"]])

In [0]:
NOTES = [
    "OOF (spec 7.6): R-fold repeat 1, 5 folds; each fold refitted from scratch. A = Pipeline(BinningProcess, unpenalised LR); "
    "B/C = common.prepare_raw on the fold rows, then the model.",
    f"Model A WoE for missing/special rows = {WOE_KW['metric_missing']} (OptBinning default, as in E9/E10); "
    "see open question [E11-woe-missing] and sheet A_missing_sensitivity.",
    f"Model C rounds: all DEV = E10 final rounds {C_FINAL_ROUNDS}; folds = stage-3 best_iter {C_BEST_ITER} x n_train / "
    f"{R1_TRAIN_SIZE:.1f} (mean R1 training size). No early stopping.",
    f"Model B monotonisation: {len(B_MONO)} features (requires expected_direction + or - in feature_dict.csv and "
    "agreement with the L1 trend).",
    "EBM shape check: term score averaged within L1 regular bins; head-tail reversal = sign(last - first bin) differs "
    f"across T1-T3 training fits. Result: {EBM_SHAPE_CHECK} ({', '.join(REVERSED) or 'no reversals'}).",
    "Points table: Scorecard pdo_odds on the uncalibrated Model A PD; higher score = lower risk. Calibration is E15.",
    "t_auc_std uses ddof=0, as in E10.",
]
meta = {"module": "E11_train", "run_time": datetime.now().isoformat(), "git_commit": GIT_COMMIT,
        "config_sha256": CONFIG_HASH}
XLSX = os.path.join(REPORTS, "04_training.xlsx")
_tmp = os.path.join(_TMP, "04_training.xlsx")
with pd.ExcelWriter(_tmp, engine="openpyxl") as w:
    pd.DataFrame([{**meta, **VERSIONS}]).to_excel(w, sheet_name="meta", index=False)
    OOF_METRICS.to_excel(w, sheet_name="oof_metrics", index=False)
    OOF_FOLD.to_excel(w, sheet_name="oof_fold_metrics", index=False)
    TFOLD.to_excel(w, sheet_name="tfold_metrics", index=False)
    TFOLD_SUMMARY.to_excel(w, sheet_name="tfold_summary", index=False)
    IN_SAMPLE.to_excel(w, sheet_name="in_sample_dev", index=False)
    A_COEF.to_excel(w, sheet_name="A_coefficients", index=False)
    A_POINTS.to_excel(w, sheet_name="A_points", index=False)
    A_BINS_VS_L1.to_excel(w, sheet_name="A_bins_vs_L1", index=False)
    A_MISSING.to_excel(w, sheet_name="A_missing_bins", index=False)
    A_SENS.to_excel(w, sheet_name="A_missing_sensitivity", index=False)
    B_MONO_DF.to_excel(w, sheet_name="B_monotonize", index=False)
    B_IMPORTANCE.to_excel(w, sheet_name="B_importance", index=False)
    B_SHAPES_FINAL.to_excel(w, sheet_name="B_shapes_final", index=False)
    B_SHAPE_BINS.to_excel(w, sheet_name="B_shape_by_L1_bin", index=False)
    B_SHAPE_CHECK.to_excel(w, sheet_name="B_shape_check", index=False)
    C_IMPORTANCE.to_excel(w, sheet_name="C_importance", index=False)
    FILE_SHA.to_excel(w, sheet_name="files_sha256", index=False)
    pd.DataFrame({"note": NOTES}).to_excel(w, sheet_name="notes", index=False)
shutil.copyfile(_tmp, XLSX)
print(f"Saved {XLSX}")

In [0]:
import mlflow
mlflow.set_experiment(cfg["databricks"]["mlflow_experiment"])
oofm = OOF_METRICS.set_index("model")
tsum = TFOLD_SUMMARY.set_index("model")
with mlflow.start_run(run_name="E11_train") as run:
    mlflow.set_tags({"module": "E11_train", "git_commit": GIT_COMMIT, "ebm_shape_check": EBM_SHAPE_CHECK})
    mlflow.log_params({"config_hash": CONFIG_HASH, "seed": SEED, "B_trial": CP["B"]["trial"], "C_trial": CP["C"]["trial"],
                       "C_final_rounds": C_FINAL_ROUNDS, "A_n_features": len(A_FEATS), "BC_n_columns": len(COLS),
                       "A_woe_metric_missing": WOE_KW["metric_missing"], "B_n_monotonized": len(B_MONO)})
    mlflow.log_metrics({**{f"{m}_oof_{k}": float(oofm.loc[m, k]) for m in "ABC" for k in ("auc", "gini", "ks")},
                        **{f"{m}_t_auc_mean": float(tsum.loc[m, "t_auc_mean"]) for m in "ABC"},
                        **{f"{m}_t_auc_std": float(tsum.loc[m, "t_auc_std"]) for m in "ABC"},
                        "A_coef_diff": A_COEF_DIFF, "A_pvalue_max": A_PMAX, "A_oof_auc_empirical_missing": AUC_A_EMP})
    for p in (XLSX, SHAPE_PNG, SHA_PATH, SAVED["model_spec.json"], SAVED["A/points_table.csv"], SAVED["A/coefficients.csv"]):
        mlflow.log_artifact(p)
    MLFLOW_RUN_ID = run.info.run_id
print(f"MLflow run: {MLFLOW_RUN_ID}")

In [0]:
# ── Acceptance (E11): re-read saved outputs from the volume and check every criterion ──
print("=" * 60); print("ACCEPTANCE CRITERIA (E11 — training and OOF)"); print("=" * 60)
oof_r = pd.read_parquet(OOF_PATH)
oof_r["app_id"] = oof_r["app_id"].astype(str)
dev_ids = set(dev_gb["app_id"])
once_per_model = {m: bool(len(g) == len(dev_gb) and g.app_id.is_unique and set(g.app_id) == dev_ids)
                  for m, g in oof_r.groupby("model")}
r1_fold_of = pd.Series({aid: int(k[1]) for k, _, va in R1 for aid in dev_gb["app_id"].iloc[va]})
fold_match = bool((oof_r.fold.to_numpy() == r1_fold_of.loc[oof_r.app_id].to_numpy()).all())
p_ok = bool(oof_r.p_oof.notna().all() and ((oof_r.p_oof > 0) & (oof_r.p_oof < 1)).all())

fold_fits = [e for e in FIT_LOG if e["fold"] in FOLD_VALID]
overlaps = {(e["model"], e["fold"]): len(e["ids"] & FOLD_VALID[e["fold"]]) for e in fold_fits}
exp_fits = 3 * (len(R1) + len(T)) + len(R1)          # A, B, C on R1 + T folds, plus the A sensitivity on R1

b2 = pickle.load(open(SAVED["B/ebm.pkl"], "rb"))
c2 = lgb.Booster(model_file=SAVED["C/lgbm.txt"])
sc2 = pickle.load(open(SAVED["A/scorecard.pkl"], "rb"))
lr2 = pickle.load(open(SAVED["A/lr_sklearn.pkl"], "rb"))
bp2 = pickle.load(open(SAVED["A/binning_process.pkl"], "rb"))
reload_ok = bool(
    np.allclose(b2.predict_proba(X_RAW_ALL)[:, 1], B_FINAL.predict_proba(X_RAW_ALL)[:, 1], atol=1e-12)
    and np.allclose(c2.predict(X_RAW_ALL), C_FINAL.predict(X_RAW_ALL), atol=1e-12)
    and np.allclose(np.asarray(sc2.score(X_A_ALL), dtype=float), A_SCORE, atol=1e-9)
    and np.allclose(lr2.predict_proba(np.asarray(bp2.transform(X_A_ALL, metric="woe", **WOE_KW), dtype=float))[:, 1],
                    p_a_full, atol=1e-12))
with open(SHA_PATH) as fh:
    sha_saved = json.load(fh)
sha_ok = bool(set(sha_saved) == set(SAVED) and all(sha256_file(SAVED[k]) == v for k, v in sha_saved.items()))
sheets = pd.ExcelFile(XLSX).sheet_names
tsum = TFOLD_SUMMARY.set_index("model")

checks = [
    ("OOF columns are app_id, model, fold, p_oof", list(oof_r.columns) == ["app_id", "model", "fold", "p_oof"]),
    (f"Every DEV GOOD+BAD app_id exactly once per model in OOF ({ {m: v for m, v in once_per_model.items()} })",
     set(once_per_model) == {"A", "B", "C"} and all(once_per_model.values())),
    ("OOF fold equals the app_id's R-fold repeat-1 fold; p_oof in (0, 1)", fold_match and p_ok),
    (f"Validation app_ids never entered any fold fit ({len(fold_fits)} fits, max overlap {max(overlaps.values())})",
     len(fold_fits) == exp_fits and max(overlaps.values()) == 0),
    (f"Model A binning fitted on training rows only (Totals count = n_train in {len(A_BIN_COUNT_CHECK)} fold fits)",
     len(A_BIN_COUNT_CHECK) == len(R1) + len(T) and all(r["ok"] for r in A_BIN_COUNT_CHECK)),
    (f"Model A coefficient signs all equal ({int((feat_coefs < 0).sum())}/{len(A_FEATS)} negative)", A_SIGNS_EQUAL),
    (f"Model A p-values < {PVALUE_MAX} (max {A_PMAX:.4g})", A_PMAX < PVALUE_MAX),
    (f"sklearn vs statsmodels max |coef diff| < 1e-4 ({A_COEF_DIFF:.2e}); Logit converged", A_COEF_DIFF < 1e-4 and A_LOGIT_CONVERGED),
    (f"Points: score = Offset + Factor ln(odds) (max diff {A_SCORE_DIFF:.1e}); higher score = lower risk "
     f"(Spearman {A_SCORE_RISK_CORR:.3f})", A_SCORE_DIFF < 1e-6 and A_SC_PROBA_DIFF < 1e-9 and A_SCORE_RISK_CORR < 0),
    (f"C trees = final rounds ({C_FINAL.num_trees()}); B main effects only",
     C_FINAL.num_trees() == C_FINAL_ROUNDS and all(len(t_) == 1 for t_ in B_FINAL.term_features_)),
    ("T-fold AUC for A, B, C on T1-T3 with t_auc_mean / t_auc_std",
     set(tsum.index) == {"A", "B", "C"} and (tsum.n_folds == len(T)).all() and tsum[["t_auc_mean", "t_auc_std"]].notna().all().all()),
    ("Saved models reload with identical predictions", reload_ok),
    (f"sha256 recorded and re-verified for {len(sha_saved)} files", sha_ok),
    (f"No OOT rows read (OOT-dated rows loaded = {N_OOT_ROWS_READ}; windows = {sorted(master_dev['window'].unique())})",
     N_OOT_ROWS_READ == 0 and set(master_dev["window"].unique()) == {"DEV"}),
    ("Report has meta sheet first", sheets[0] == "meta"),
]
all_pass = all(ok for _, ok in checks)
for desc, ok in checks:
    status = "\u2705 PASS" if ok else "\u274c FAIL"
    print(f"  {status} : {desc}")
print(f"\n  INFO : ebm_shape_check = {EBM_SHAPE_CHECK} (reversals: {REVERSED or 'none'})")

summary = {"module": "E11_train", "run_time": datetime.now().isoformat(), "elapsed_s": round(time.time() - _t0, 1),
           "git_commit": GIT_COMMIT, "config_hash": CONFIG_HASH, "rows_dev_good_bad": len(dev_gb),
           "bads_dev": int(y.sum()), "oot_rows_read": N_OOT_ROWS_READ, "oof_rows": len(oof_r),
           **{f"{m}_oof_auc": round(float(oofm.loc[m, "auc"]), 4) for m in "ABC"},
           **{f"{m}_oof_gini": round(float(oofm.loc[m, "gini"]), 4) for m in "ABC"},
           **{f"{m}_oof_ks": round(float(oofm.loc[m, "ks"]), 4) for m in "ABC"},
           **{f"{m}_t_auc_mean": round(float(tsum.loc[m, "t_auc_mean"]), 4) for m in "ABC"},
           **{f"{m}_t_auc_std": round(float(tsum.loc[m, "t_auc_std"]), 4) for m in "ABC"},
           "A_coef_diff": A_COEF_DIFF, "A_pvalue_max": A_PMAX, "ebm_shape_check": EBM_SHAPE_CHECK,
           "ebm_reversals": REVERSED, "B_monotonized": len(B_MONO), "C_final_rounds": C_FINAL_ROUNDS,
           "mlflow_run_id": MLFLOW_RUN_ID, "pass_fail": "PASS" if all_pass else "FAIL"}
with open(os.path.join(RUN_SUMS, "s04_train.json"), "w") as fh:
    json.dump(summary, fh, indent=2, default=str)
print(f"\nRun summary: {RUN_SUMS}/s04_train.json")
for desc, ok in checks:
    assert ok, f"ACCEPTANCE FAILED: {desc}"
print("\n\u2705 All E11 acceptance criteria passed.")

In [0]:
# ── Compliance check (not part of E11 outputs; nothing saved). aecb_AECB_Score is compliance-rejected and dropped
#    at L0 (spec 6.1), so it must not reach the L4 pool, Model A or the fitted B/C models. The original drop-one
#    sensitivity (refit A and C without the feature) only runs if the feature is still in the pool. ──
DROP_FEAT = "aecb_AECB_Score"
_fl = spark.table(f"{cat}.{sch}.acard_feature_log").filter(f"feature = '{DROP_FEAT}'").toPandas()
assert len(_fl) == 1, f"{DROP_FEAT}: expected 1 feature-log row, found {len(_fl)}"
print(f"Feature log: L0_pass={_fl['L0_pass'].iat[0]}, dropped_at={_fl['dropped_at'].iat[0]}, "
      f"drop_rule={_fl['drop_rule'].iat[0]}")
COMPLIANCE_EXCLUDED = DROP_FEAT not in POOL
if COMPLIANCE_EXCLUDED:
    assert _fl["L0_pass"].iat[0] == "N" and _fl["dropped_at"].iat[0] == "L0", f"{DROP_FEAT} not dropped at L0"
    assert DROP_FEAT not in A_FEATS, f"{DROP_FEAT} still in Model A"
    assert not any(c == DROP_FEAT or c.startswith(DROP_FEAT + "_") for c in COLS), f"{DROP_FEAT} in B/C columns"
    assert not any(DROP_FEAT in t_ for t_ in B_FINAL.term_names_), f"{DROP_FEAT} in Model B terms"
    assert not any(DROP_FEAT in c for c in C_FINAL.feature_name()), f"{DROP_FEAT} in Model C features"
    assert not os.path.exists(os.path.join(BIN_DIR, f"{DROP_FEAT}.pkl")), f"stale L1 binning for {DROP_FEAT}"
    print(f"\u2705 {DROP_FEAT} is compliance-excluded: absent from L4 pool ({len(POOL)}), Model A ({len(A_FEATS)} features), "
          f"B/C columns ({len(COLS)}), fitted B and C, and binning/L1. Drop-one sensitivity skipped (nothing to drop).")

A_FEATS_X = [f for f in A_FEATS if f != DROP_FEAT]
POOL_X = [f for f in POOL if f != DROP_FEAT]
print(f"{DROP_FEAT} in Model A: {DROP_FEAT in A_FEATS} | A features {len(A_FEATS)} -> {len(A_FEATS_X)} | "
      f"C pool {len(POOL)} -> {len(POOL_X)}")

def fit_a_s(idx, feats):
    params = {f: A_FIT_PARAMS[f] for f in feats}
    cats = [f for f in A_CAT if f in feats] or None
    bp = BinningProcess(variable_names=feats, categorical_variables=cats, binning_fit_params=params)
    pipe = Pipeline([("binning", bp), ("lr", common.lr_unpenalised(max_iter=2000))])
    return pipe.fit(x_a(idx)[feats], y.iloc[idx])

def c_inputs(pool):
    cols = list(common.prepare_raw(dev_gb.iloc[:5], pool, fdict).columns)
    mono = common.monotone_vector(cols, pool, binning_objects, fdict)
    return cols, mono, [c for c in CAT_COLS if c in cols]

def fit_c_s(idx, pool, rounds):
    cols, mono, cats = c_inputs(pool)
    ds = lgb.Dataset(common.prepare_raw(dev_gb.iloc[idx], pool, fdict)[cols], label=y.iloc[idx],
                     categorical_feature=cats, free_raw_data=False,
                     params={"feature_pre_filter": False, "verbose": -1})
    return lgb.train({**C_PARAMS, "monotone_constraints": mono}, ds, num_boost_round=rounds), cols

def pred_c_s(m, cols, idx, pool):
    return m.predict(common.prepare_raw(dev_gb.iloc[idx], pool, fdict)[cols])

VARIANTS = {"A_full": ("A", A_FEATS), "A_no_" + DROP_FEAT: ("A", A_FEATS_X),
            "C_full": ("C", POOL), "C_no_" + DROP_FEAT: ("C", POOL_X)}

def run_variant(kind, feats, tr, va):
    if kind == "A":
        m = fit_a_s(tr, feats)
        return m.predict_proba(x_a(va)[feats])[:, 1]
    m, cols = fit_c_s(tr, feats, c_rounds(len(tr)))
    return pred_c_s(m, cols, va, feats)

if not COMPLIANCE_EXCLUDED:
    t = time.time()
    sens_rows, sens_t_rows = [], []
    for name, (kind, feats) in VARIANTS.items():
        p_oof = np.full(len(dev_gb), np.nan)
        for _, tr, va in R1:
            p_oof[va] = run_variant(kind, feats, tr, va)
        assert not np.isnan(p_oof).any()
        sens_rows.append({"variant": name, "n_features": len(feats), **perf(y, p_oof)})
        for tname, tr, va in T:
            sens_t_rows.append({"variant": name, "tfold": tname, "auc": roc_auc_score(y.iloc[va], run_variant(kind, feats, tr, va))})

    SENS_DROP = pd.DataFrame(sens_rows).set_index("variant")
    t_piv = pd.DataFrame(sens_t_rows).pivot(index="variant", columns="tfold", values="auc")
    SENS_DROP = SENS_DROP.join(t_piv).assign(t_auc_mean=t_piv.mean(axis=1), t_auc_std=t_piv.std(axis=1, ddof=0))
    SENS_DROP.columns = [f"oof_{c}" if c in ("auc", "gini", "ks") else (f"t_auc_{c}" if c in t_piv.columns else c)
                         for c in SENS_DROP.columns]

    # Full-model rows must reproduce the E11 results above (check the sensitivity code is faithful)
    oofm_chk = OOF_METRICS.set_index("model")
    for m in "AC":
        d = abs(SENS_DROP.loc[f"{m}_full", "oof_auc"] - oofm_chk.loc[m, "auc"])
        print(f"Check {m}_full OOF AUC vs E11: |diff| = {d:.2e}")

    DELTA = pd.DataFrame({m: SENS_DROP.loc[f"{m}_no_{DROP_FEAT}"] - SENS_DROP.loc[f"{m}_full"] for m in "AC"}).T
    DELTA = DELTA.drop(columns="n_features").add_prefix("delta_")
    print(f"Sensitivity runtime {time.time() - t:.0f}s")
    display(SENS_DROP.round(4).reset_index())
    display(DELTA[["delta_oof_auc", "delta_oof_gini", "delta_oof_ks", "delta_t_auc_mean"]].round(4).reset_index(names="model"))

    # Model A without the feature, refit on all DEV: are signs / p-values still within the L5 rules?
    ma_x = fit_a_s(ALL, A_FEATS_X)
    Xw = pd.DataFrame(np.asarray(ma_x.named_steps["binning"].transform(x_a(ALL)[A_FEATS_X], metric="woe", **WOE_KW),
                                 dtype=float), columns=A_FEATS_X)
    logit_x = sm.Logit(y.to_numpy(), sm.add_constant(Xw, has_constant="add")).fit(disp=0, maxiter=200)
    coef_cmp = pd.DataFrame({"coef_full": A_LOGIT.params, "coef_no_feat": logit_x.params,
                             "p_full": A_LOGIT.pvalues, "p_no_feat": logit_x.pvalues})
    print(f"Model A without {DROP_FEAT}: all coefs negative = {bool((logit_x.params[A_FEATS_X] < 0).all())}, "
          f"max p = {logit_x.pvalues[A_FEATS_X].max():.4g} (limit {PVALUE_MAX})")
    display(coef_cmp.round(4).reset_index(names="term"))